In [0]:
# ── Cell 1 — imports ───────────────────────────────────────
from pyspark import pipelines as dp
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_unixtime, current_timestamp

In [0]:
# ── Cell 2 — schema ────────────────────────────────────────
wiki_schema = StructType([
    StructField("id",          LongType(),    True),
    StructField("type",        StringType(),  True),
    StructField("title",       StringType(),  True),
    StructField("user",        StringType(),  True),
    StructField("bot",         BooleanType(), True),
    StructField("timestamp",   LongType(),    True),
    StructField("server_name", StringType(),  True),
    StructField("wiki",        StringType(),  True),
    StructField("namespace",   IntegerType(), True),
    StructField("meta", StructType([
        StructField("domain", StringType(), True),
        StructField("dt",     StringType(), True),
        StructField("id",     StringType(), True),
    ]), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True),
    ]), True),
])

In [0]:
# ── Cell 3 — Bronze streaming table ────────────────────────
RAW = "/Volumes/streaming/landing/inbound/raw/"

@dp.table(
    name             = "bronze_wiki_edits",
    comment          = "Raw Wikipedia edit events from Volume landing zone",
    table_properties = {"quality": "bronze"},
)
@dp.expect("valid_event_id", "event_id IS NOT NULL")
@dp.expect("valid_wiki",     "wiki IS NOT NULL")
def bronze_wiki_edits():
    return (
        spark.readStream
        .format("json")
        .schema(wiki_schema)
        .option("maxFilesPerTrigger", 5)
        .load(RAW)
        .select(
            col("id").alias("event_id"),
            col("type").alias("event_type"),
            col("title"),
            col("user"),
            col("bot"),
            col("wiki"),
            col("namespace"),
            col("server_name").alias("domain"),
            col("meta.id").alias("meta_id"),
            from_unixtime(col("timestamp")).cast("timestamp").alias("event_time"),
            current_timestamp().alias("ingested_at"),
            col("length.old").alias("len_old"),
            col("length.new").alias("len_new"),
        )
    )